# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [ ]:
VARIANT = 2   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [ ]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [ ]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | … |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | … |
| группы и доли | … |
| основная метрика | … |
| метрики-стражи: что не должно ухудшиться | … |
| α, мощность, MDE — числами из ячейки выше | … |
| поправка на множественные сравнения | … |
| оценщик | … |
| окно | … |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | … |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [ ]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
alpha = 0.05

print(f"SRM: p = {answer_1:.3f}")
print(f"Баланс групп до эксперимента: p = {answer_2:.3f}")

print(f"\nMDE без CUPED: {mde_raw:.2f} ₽")
print(f"MDE с CUPED: {mde_cuped:.2f} ₽")
print(f"Целевой рост на 3%: {three_pct:.2f} ₽")

print(f"\nЭффект: {effect['diff']:.2f} ₽")
print(f"95% ДИ: [{effect['lo']:.2f}; {effect['hi']:.2f}] ₽")
print(f"p-value: {effect['p']:.4f}")

relative_effect = effect["diff"] / control["spend"].mean() * 100

print(f"Относительный эффект: {relative_effect:.2f}%")
print(f"Эффект составляет {abs(effect['diff']) / mde_raw:.1%} от MDE")

print(f"Интервал включает ноль: {effect['lo'] <= 0 <= effect['hi']}")

print(f"\nt-критерий: p = {answer_4[0]:.4f}")
print(f"U-критерий: p = {answer_4[1]:.4f}")

print(f"t-критерий значим: {answer_4[0] < alpha}")
print(f"U-критерий значим: {answer_4[1] < alpha}")

#### ✍️ Ответ на вопрос 1

По результатам проверок проблем с разбиением на группы нет: p-значение SRM равно 0,776, а проверки баланса групп до эксперимента — 0,940, оба значения выше 0,05. Оборот в тесте вырос на 76,94 руб. (2,31 %), что меньше MDE как без CUPED (240,34 руб.), так и с ним (192,50 руб.). Доверительный интервал от −82,94 до 236,81 руб. включает ноль, поэтому уверенно говорить о росте нельзя. Оба критерия дали одинаковый вывод (p = 0,3787 для t-критерия и 0,0613 для U-критерия), что критерий не значим

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [ ]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [ ]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [ ]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [ ]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print("Метрики-стражи:")
print(f"Изменение оттока: {answer_5[0]:+.2f} п.п.")
print(f"p-value оттока: {p_churn:.4f}")
print(f"Изменение обращений: {answer_5[2]:+.3f}")
print(f"p-value обращений: {guards.loc['tickets', 'p']:.4f}")

print(f"\nСработал страж оттока: {p_churn < alpha}")
print(f"Сработал страж обращений: {guards.loc['tickets', 'p'] < alpha}")

print("\nПоправка Холма:")
print(metrics[["feature", "TTest p-value", "holm", "rejected"]])
print(f"Значимые метрики: {answer_6}")

# Сравнение ширины доверительных интервалов
width_raw = effect["hi"] - effect["lo"]
width_cuped = effect_cuped["hi"] - effect_cuped["lo"]

ci_reduction = (1 - width_cuped / width_raw) * 100
effect_change = effect_cuped["diff"] - effect["diff"]

print(f"\nШирина интервала без CUPED: {width_raw:.2f} ₽")
print(f"Ширина интервала с CUPED: {width_cuped:.2f} ₽")
print(f"Сужение интервала: {ci_reduction:.2f}%")
print(f"Снижение дисперсии: {reduction:.2%}")

print(f"\nЭффект без CUPED: {effect['diff']:.2f} ₽")
print(f"Эффект с CUPED: {effect_cuped['diff']:.2f} ₽")
print(f"Изменение оценки эффекта: {effect_change:+.2f} ₽")

# Сравнение оборота до эксперимента
prev_test = test["spend_prev"].mean()
prev_control = control["spend_prev"].mean()
prev_diff = prev_test - prev_control

print(f"\nСредний оборот прошлого периода:")
print(f"Test: {prev_test:.2f} ₽")
print(f"Control: {prev_control:.2f} ₽")
print(f"Разница: {prev_diff:+.2f} ₽")

print(f"Коэффициент theta: {theta:.3f}")
print(f"Поправка CUPED: {-theta * prev_diff:+.2f} ₽")

#### ✍️ Ответ на вопрос 2

Отток вырос на 0,59 п.п. (p = 0,222), а число обращений — на 0,003 на клиента; по результатам проверки в задаче 6 значимых ухудшений нет, и после поправки Холма ни одна метрика не осталась значимой. CUPED снизил дисперсию на 38,4 % и сузил интервал на 10,20 % — с 319,75 до 287,13 руб.. До эксперимента клиенты теста тратили в среднем 3225,07 руб., а контроля — 3231,53 руб., поэтому поправка на исходное различие увеличила оценку эффекта с 76,94 до 81,02 руб..

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [ ]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.

rng = np.random.default_rng(VARIANT)
x_test = test["spend"].to_numpy()
x_control = control["spend"].to_numpy()
boot = np.empty(2000)
for i in range(2000):
    boot[i] = (rng.choice(x_test, size=len(x_test)).mean()
               - rng.choice(x_control, size=len(x_control)).mean())
answer_9 = [round(float(v), 2) for v in np.percentile(boot, [2.5, 97.5])]
answer_9


### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [ ]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.

covariates = mine[["spend_prev", "spend_prev2", "purchases_prev", "sessions_prev", "sessions_prev2"]].copy()
covariates["spend_prev+spend_prev2"] = mine["spend_prev"] + mine["spend_prev2"]
r2 = covariates.corrwith(mine["spend"]).pow(2) * 100
r2_post = mine["purchases"].corr(mine["spend"]) ** 2 * 100
print(r2.sort_values(ascending=False).round(1))
print(f"purchases (после воздействия): {r2_post:.1f}%")
answer_10 = [str(r2.idxmax()), round(float(r2.max()), 1)]
answer_10

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.

feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]

def match_table(data, features):
    data = data.assign(t=(data["grp"] == "test").astype(int))
    roles = {"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
             **{f: FeatureRole() for f in features}}
    return F(Matching(n_neighbors=5).execute(Dataset(roles=roles, data=data, default_role=InfoRole())).resume)

matching = match_table(mine, feats)
print(matching.to_string())

def matching_effect(table, estimand):
    # HypEx: строка ATE/ATT и столбцы оценки и границ интервала.
    # Нормализуем подписи, чтобы учесть регистр и пробелы.
    def norm(s):
        return "".join(ch for ch in str(s).lower() if ch.isalnum())
    tbl = table.reset_index()
    mask = tbl.apply(lambda col: col.astype(str).str.strip().str.upper().eq(estimand), axis=0).any(axis=1)
    if not mask.any():
        raise ValueError(f"Не найдена строка {estimand}: {table}")
    row = tbl.loc[mask].iloc[0]
    cols = {norm(c): c for c in tbl.columns}
    def val(names):
        for name in names:
            if norm(name) in cols:
                return float(row[cols[norm(name)]])
        raise ValueError(f"Не найден столбец {names}; доступны {list(table.columns)}")
    return [val(["effect", "estimate", "difference", "value", "mean", "effect size"]),
            val(["ci lower", "ci low", "lower ci", "lower", "lower bound", "ci left", "left conf int", "conf int lower", "2.5%"]),
            val(["ci upper", "ci high", "upper ci", "upper", "upper bound", "ci right", "right conf int", "conf int upper", "97.5%"])]

answer_11 = [round(float(v), 2) for v in matching_effect(matching, "ATE")]
answer_11


### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [ ]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽

commission_income = effect_cuped["diff"] * COMMISSION
ticket_expense = guards.loc["tickets", "diff"] * TICKET_COST
churn_expense = guards.loc["churned", "diff"] * CHURN_COST
net = commission_income - ticket_expense - churn_expense
print(f"Комиссия: {commission_income:.2f} ₽; обращения: {-ticket_expense:+.2f} ₽; отток: {-churn_expense:+.2f} ₽; итог: {net:.2f} ₽")
answer_12 = round(float(net), 2)
answer_12

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
# Сравнение оценок эффекта и доверительных интервалов
comparison = pd.DataFrame(
    [
        ["Без корректировки", *answer_3],
        ["CUPED", *answer_7[2:]],
        ["Бутстреп", effect["diff"], *answer_9],
        ["Мэтчинг (ATE)", *answer_11],
    ],
    columns=["method", "effect", "lo", "hi"]
).set_index("method")

comparison["width"] = comparison["hi"] - comparison["lo"]

comparison["reduction_pct"] = (
    1 - comparison["width"] / comparison.loc["Без корректировки", "width"]
) * 100

comparison["significant"] = (
    (comparison["lo"] > 0) | (comparison["hi"] < 0)
)

print("Сравнение методов:")
print(comparison.round(2).to_string())

print(f"\nСамый узкий интервал: {comparison['width'].idxmin()}")

# Сравнение ковариат
print("\nКовариаты и потенциальное снижение дисперсии:")
print(r2.sort_values(ascending=False).round(1).to_string())

print(f"\nЛучшая допустимая ковариата: {answer_10[0]}")
print(f"Её rho²: {answer_10[1]:.1f}%")
print(f"rho² для purchases: {r2_post:.1f}%")

# Экономический результат
print("\nЭкономика акции на одного клиента:")

print(f"Дополнительная комиссия: {commission_income:+.2f} ₽")
print(f"Расходы на обращения: {-ticket_expense:+.2f} ₽")
print(f"Потери от оттока: {-churn_expense:+.2f} ₽")
print(f"Итоговый результат: {net:+.2f} ₽")
print(f"Акция окупается: {net > 0}")

# Эффект, необходимый для безубыточности
break_even_effect = (ticket_expense + churn_expense) / COMMISSION

print(f"\nНеобходимый прирост оборота: {break_even_effect:.2f} ₽")
print(f"Фактический прирост оборота: {effect_cuped['diff']:.2f} ₽")

#### ✍️ Ответ на вопрос 3

Я скорее доверяю оценке CUPED (81,02 руб.): эксперимент рандомизирован, а поправка на оборот до акции позволяет учесть различия между клиентами, не меняя состав групп. Ширина обычного интервала составляет 319,75 руб., после CUPED — 287,13 руб., у бутстрепа — 343,49 руб., у мэтчинга — 280,12 руб.: бутстреп учитывает эмпирическое распределение трат, а мэтчинг сравнивает похожих клиентов, но более узкий интервал сам по себе не делает его оценку надёжнее; все интервалы включают ноль. Хотя у purchases самое высокое ρ² (71,6 %), покупки совершались уже во время акции и могли зависеть от неё, поэтому лучше использовать признаки предпериода, например sessions_prev (43,7 %). При заданных допущениях акция не окупается: на клиента приходится 1,22 руб. дополнительной комиссии, 0,46 руб. расходов на поддержку и 4,43 руб. потерь от оттока, итог — минус 3,67 руб., причём в расчёте ещё нет стоимости кэшбэка, расходов на проведение акции, долгосрочных последствий и оценки неопределённости финансового результата.

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:** хотели выяснить, увеличит ли кэшбэк 5 % на каршеринг средний оборот клиентов в партнёрских сервисах на 3 % за четыре недели. В эксперименте участвовали 6000 клиентов. Проверки размеров групп и оборота до акции не выявили явных проблем с разбиением.

**Что получилось:** по основной оценке с CUPED оборот вырос на 81,02 руб. на клиента, но доверительный интервал составляет от −62,55 до 224,58 руб.. Это значит, что пока нельзя уверенно говорить о положительном эффекте. Отток вырос на 0,59 п.п., обращения в поддержку — на 0,003 на клиента, однако после поправки на множественные сравнения статистически значимых изменений не обнаружено.

**Какие остались вопросы:** выборка недостаточно чувствительна к целевому росту на 3 % (96,85 руб.), поскольку MDE даже с CUPED составляет 192,50 руб.. По нашим предположениям акция приносит убыток 3,67 руб. на клиента за четыре недели, ещё без учёта стоимости самого кэшбэка. 

**Что рекомендуем:** пока не запускать акцию на всех. Стоит пересмотреть условия кэшбэка и его экономику, затем провести повторный эксперимент с большей выборкой, если расчёты покажут, что акция может быть прибыльной. Текущих данных недостаточно, чтобы подтвердить нужный рост оборота и оправдать масштабирование.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [ ]:
# Исследование А: ваш код. Всё — от mine.
# Исследование А. Без рандомизации

# Определяем активных клиентов по медиане всего варианта
median_sessions = mine["sessions_prev"].median()

active_test = mine[
    (mine["grp"] == "test")
    & (mine["sessions_prev"] >= median_sessions)
]

active_control = mine[
    (mine["grp"] == "control")
    & (mine["sessions_prev"] >= median_sessions)
]

full_control = mine[mine["grp"] == "control"]

# Формируем наблюдательные данные
observational = pd.concat(
    [active_test, full_control],
    ignore_index=True
)

print(f"Медиана sessions_prev: {median_sessions}")
print(f"Активных клиентов теста: {len(active_test)}")
print(f"Активных клиентов контроля: {len(active_control)}")
print(f"Всего клиентов контроля: {len(full_control)}")

# 1. Наивная разность средних
naive = (
    active_test["spend"].mean()
    - full_control["spend"].mean()
)

# 2. Мэтчинг со всеми признаками задачи 11
features_a = [
    "age",
    "tenure_days",
    "kyc_level",
    "members",
    "spend_prev",
    "sessions_prev",
]

matching_a = match_table(observational, features_a)

att, att_lo, att_hi = matching_effect(matching_a, "ATT")

# 3. Эталонная оценка по рандомизированным данным
reference = (
    active_test["spend"].mean()
    - active_control["spend"].mean()
)

# 4. Мэтчинг без sessions_prev
features_without_sessions = [
    f for f in features_a if f != "sessions_prev"
]

matching_without_sessions = match_table(
    observational,
    features_without_sessions
)

att_without, att_without_lo, att_without_hi = matching_effect(
    matching_without_sessions, "ATT"
)

# 5. Сравниваем результаты
results_a = pd.DataFrame({
    "Метод": [
        "Наивная разность",
        "Мэтчинг (ATT)",
        "Эталон",
        "Мэтчинг без sessions_prev",
    ],
    "Эффект, руб.": [
        naive,
        att,
        reference,
        att_without,
    ],
}).set_index("Метод")

results_a["Отклонение от эталона, руб."] = (
    results_a["Эффект, руб."] - reference
)

results_a["Абсолютное отклонение, руб."] = (
    results_a["Отклонение от эталона, руб."].abs()
)

print("\nСравнение оценок:")
display(results_a.round(2))

print(f"ATT: {att:.2f} руб.")
print(f"95% ДИ: [{att_lo:.2f}; {att_hi:.2f}]")

print(f"\nATT без sessions_prev: {att_without:.2f} руб.")
print(f"95% ДИ: [{att_without_lo:.2f}; {att_without_hi:.2f}]")

print(f"\nИзменение ATT после исключения sessions_prev: "
      f"{att_without - att:+.2f} руб.")

import matplotlib.pyplot as plt

ax = results_a["Эффект, руб."].plot.barh(
    figsize=(9, 4)
)

ax.axvline(
    reference,
    linestyle="--",
    label="Эталон"
)

ax.set_xlabel("Оценка эффекта, руб.")
ax.set_ylabel("")
ax.set_title("Сравнение методов оценки эффекта")
ax.legend()
ax.bar_label(ax.containers[0], fmt="%.2f")

plt.tight_layout()
plt.show()

#### ✍️ Исследование А

В рамках исследования активными считали клиентов, у которых количество сессий в предыдущем периоде было не меньше медианы (15 сессий). В наблюдательную выборку включили 1549 активных клиентов тестовой группы и всех 3011 клиентов контрольной группы. Посчитали обычную разность средних и оценку ATT с помощью мэтчинга по шести признакам. В качестве эталона использовали разность средних между 1549 активными клиентами теста и 1506 активными клиентами контроля. Дополнительно повторили мэтчинг, исключив признак sessions_prev.

При обычном сравнении средних эффект составил 1475,00 руб., тогда как эталонная оценка — всего 31,80 руб. Мэтчинг позволил приблизиться к эталону: оценка ATT составила 35,22 руб. с доверительным интервалом от −179,59 до 250,03 руб., а отклонение от эталона — всего 3,42 руб. После исключения sessions_prev оценка выросла на 467,93 руб. и составила 503,15 руб. с доверительным интервалом от 281,54 до 724,75 руб.

При обычном сравнении средних можно ошибочно учесть часть эффекта, связанного с исходной активностью клиентов. Мэтчинг позволил существенно уменьшить это смещение, однако исключение признака, по которому отбирались активные клиенты, снова привело к завышению эффекта и даже появлению статистически значимого результата. Мэтчингу можно доверять, когда учтены существенные факторы отбора и в контрольной группе есть сопоставимые клиенты. При этом исследование не доказывает, что мэтчинг всегда позволяет получить несмещённую оценку: в реальных наблюдательных данных могут присутствовать неучтённые факторы, а сама эталонная оценка также подвержена случайной погрешности.

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [ ]:
# Исследование Б: ваш код. Всё — от mine.
# Исследование Б. Сколько клиентов было нужно

# Используем только контрольную группу
control_b = mine.loc[mine["grp"] == "control"]

y_raw = control_b["spend"]

# CUPED: theta рассчитываем только по контрольной группе
y_cuped, theta_b = cuped(
    y_raw,
    control_b["spend_prev"]
)

# Эффект — 3% от среднего оборота контроля
delta_b = 0.03 * y_raw.mean()

alpha_b = 0.05

# Включаем также фактический размер контрольной группы
n_values = sorted(set([
    1000, 3000, 10000, 20000, 40000,
    len(control_b)
]))


def simulated_power(y, n, effect, k=500):
    """Оценка мощности через симуляцию экспериментов."""
    rng = np.random.default_rng(VARIANT)
    y = np.asarray(y)

    hits = 0

    for _ in range(k):
        # Выборка с возвращением
        sample = rng.choice(
            y, size=2 * n, replace=True
        )

        # В первую группу вносим искусственный эффект
        a = sample[:n] + effect
        b = sample[n:]

        # t-критерий Уэлча
        p = stats.ttest_ind(
            a, b, equal_var=False
        ).pvalue

        hits += p < alpha_b

    return hits / k


def theoretical_power(sigma, n, effect):
    """Мощность по формуле из семинара С7."""
    z_alpha = stats.norm.ppf(1 - alpha_b / 2)

    return stats.norm.cdf(
        effect / (sigma * np.sqrt(2 / n)) - z_alpha
    )


# Считаем мощность четырьмя способами
results_b = pd.DataFrame([
    {
        "n": n,

        "Симуляция без CUPED":
            simulated_power(y_raw, n, delta_b),

        "Формула без CUPED":
            theoretical_power(y_raw.std(), n, delta_b),

        "Симуляция с CUPED":
            simulated_power(y_cuped, n, delta_b),

        "Формула с CUPED":
            theoretical_power(y_cuped.std(), n, delta_b),
    }
    for n in n_values
]).set_index("n")

print(f"Эффект +3%: {delta_b:.2f} руб.")
print(f"Коэффициент theta: {theta_b:.3f}")

print(f"Снижение дисперсии: "
      f"{1 - y_cuped.var() / y_raw.var():.1%}")

display(results_b.style.format("{:.1%}"))

# Определяем необходимый размер выборки
for col, y in [
    ("Симуляция без CUPED", y_raw),
    ("Симуляция с CUPED", y_cuped),
]:
    reached = results_b.index[results_b[col] >= 0.8]

    print(f"\n{col}:")

    print(
        "Минимальный проверенный n с мощностью >= 80%:",
        min(reached) if len(reached) else "не достигнута"
    )

    print(
        "Необходимое n по формуле:",
        n_per_group(y.std(), delta_b)
    )

# Проверяем мощность при фактическом размере группы
print(f"\nМощность при размере контрольной группы "
      f"({len(control_b)} клиентов):")

display(
    results_b.loc[[len(control_b)]].style.format("{:.1%}")
)

import matplotlib.pyplot as plt

ax = results_b.plot(
    marker="o",
    figsize=(10, 5)
)

ax.axhline(
    0.8,
    linestyle="--",
    color="gray",
    label="Целевая мощность 80%"
)

ax.set_xlabel("Количество клиентов на группу")
ax.set_ylabel("Статистическая мощность")
ax.set_title("Мощность эксперимента при росте оборота на 3%")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

#### ✍️ Исследование Б

В рамках анализа использовали оборот клиентов контрольной группы и задали искусственный эффект в размере 3% от среднего оборота, что составило 99,90 руб. Для каждого размера выборки провели 500 симуляций со случайным отбором клиентов с возвращением и сравнением групп с помощью t-критерия Уэлча при уровне значимости 0,05. Затем повторили расчёты для оборота после CUPED с использованием оборота предыдущего периода в качестве ковариаты и сравнили полученную мощность с теоретической.

CUPED позволил снизить дисперсию оборота на 41,2%. При фактическом размере контрольной группы (3011 клиентов) эмпирическая мощность составила 21,0% без CUPED и 34,2% с CUPED, тогда как теоретические значения составили 20,0% и 30,7% соответственно. При 10 000 клиентов на группу мощность достигла 54,0% без CUPED и 72,8% с CUPED, а при 20 000 — 82,8% и 96,4% соответственно. Таким образом, среди проверенных размеров выборки необходимая мощность в обоих случаях была достигнута при 20 000 клиентов на группу. Однако теоретический расчёт показал, что без CUPED необходимо 18 922 клиента на группу, а с CUPED — 11 135.

Фактического количества участников эксперимента было недостаточно для надёжного обнаружения роста оборота на 3% даже при использовании CUPED. При этом CUPED позволяет существенно повысить мощность эксперимента и сократить необходимый размер выборки. Небольшие расхождения можно объяснить тяжёлым хвостом распределения оборота, использованием нормального приближения в формуле и ограниченным количеством симуляций. Исследование не доказывает, что акция действительно увеличивает оборот на 3%, а лишь показывает, сколько клиентов потребуется для обнаружения такого эффекта при сохранении текущих характеристик распределения оборота.